## Read-in raw files and down-sample

In [2]:
import os
import glob
import pandas as pd
import re

In [12]:
dft = pd.read_csv('./restaurant-ratings/west-south-central.csv')

In [15]:
dft['rating'].value_counts()

rating
5.0    8411777
4.0    2940584
3.0    1391761
1.0    1131868
2.0     721506
Name: count, dtype: int64

### List of files to process

In [5]:
path = "./restaurant-ratings/" # Read-from and write-to this directory

file_list = glob.glob(os.path.join(path, '*.csv'))
file_list

['./restaurant-ratings/mid-atlantic.csv',
 './restaurant-ratings/south-atlantic.csv',
 './restaurant-ratings/west-north-central.csv',
 './restaurant-ratings/east-north-central.csv',
 './restaurant-ratings/west-south-central.csv',
 './restaurant-ratings/east-south-central.csv',
 './restaurant-ratings/mountain.csv',
 './restaurant-ratings/pacific.csv',
 './restaurant-ratings/new-england.csv']

### Temporarily pick 2

In [105]:
file_list = file_list[6], file_list[10] # Test with a small file, Alaska
file_list

('/Users/andy/_data/Google Local/review-Alaska_10.json.gz',
 '/Users/andy/_data/Google Local/review-District_of_Columbia_10.json.gz')

### Stratify down-sample 

- Tried 2%, too big, back to 1%
  
  

And save to same folder as .csv

In [22]:
i = 1

for file in file_list:
    print(f' "{file}" ')
    #df = pd.read_json(file, compression='gzip', lines=True, dtype={'text': 'str'})
    df = pd.read_csv(file) #DC
        
    # Clean up
    df = df[['rating','text']] # Keep relevant columns
    df = df[df['text'].apply(lambda x: isinstance(x, str))] # Drop non-string instances
    df = df[ df['text'] != 'None' ] # Drop blank entries

    # Calculate stratified sample sizes by rating class
    sample_size = int(round(df.shape[0] * 0.01)) # sample size DC
    group_counts = df['rating'].value_counts(normalize=True)
    group_samples = (group_counts * sample_size).astype(int)
    group_samples # Number of samaples to draw for each rating class

    # Stratified sample by rating class
    df_list = []
    for strata in range(1, int(df['rating'].max())+1): # Error was raised here for some of dfs. Some had ratings as floats
        df_strata = df[ df['rating'] == strata ]
        df_strata = df_strata.sample(group_samples[strata], random_state=100)
        df_list.append(df_strata)

    df_sampled = pd.concat(df_list, ignore_index=True)

    print(f'Sampled: {df_sampled.shape[0]} \nOriginal: {df.shape[0]}') #I think these were flipped

    #state_name = re.search(r'review-(.+?)_10', file).group(1)
    df_sampled.to_csv(f'{path}/{i}.csv.gz', index=False, compression='gzip')
    i += 1 #DC

 "./restaurant-ratings/mid-atlantic.csv" 
Sampled: 94234 
Original: 9423700
 "./restaurant-ratings/south-atlantic.csv" 
Sampled: 195340 
Original: 19534173
 "./restaurant-ratings/west-north-central.csv" 
Sampled: 54595 
Original: 5459823
 "./restaurant-ratings/east-north-central.csv" 
Sampled: 136193 
Original: 13619515
 "./restaurant-ratings/west-south-central.csv" 
Sampled: 145970 
Original: 14597295
 "./restaurant-ratings/east-south-central.csv" 
Sampled: 52974 
Original: 5297700
 "./restaurant-ratings/mountain.csv" 
Sampled: 90188 
Original: 9019109
 "./restaurant-ratings/pacific.csv" 
Sampled: 158613 
Original: 15861525
 "./restaurant-ratings/new-england.csv" 
Sampled: 28734 
Original: 2873701


### Read-in saved .csv files

In [23]:
file_list_csv = glob.glob(os.path.join(path, '*.csv.gz'))
file_list_csv

['./restaurant-ratings/3.csv.gz',
 './restaurant-ratings/2.csv.gz',
 './restaurant-ratings/6.csv.gz',
 './restaurant-ratings/9.csv.gz',
 './restaurant-ratings/7.csv.gz',
 './restaurant-ratings/4.csv.gz',
 './restaurant-ratings/5.csv.gz',
 './restaurant-ratings/8.csv.gz',
 './restaurant-ratings/1.csv.gz']

In [24]:
df_csv_list = []
for file in file_list_csv:
    file = file_list_csv[1]
    df_csv = pd.read_csv(file)
    df_csv_list.append(df_csv)

df_all_states = pd.concat(df_csv_list, ignore_index=True)


In [34]:
from sklearn.model_selection import train_test_split

df_train, df_temp = train_test_split(df_all_states, train_size=.7, stratify=df_all_states['rating'])

df_val, df_test = train_test_split(df_temp, train_size=.5, stratify=df_temp['rating'])

In [37]:
print('proportion counts')
print('train', df_train['rating'].value_counts(normalize=True))
print('val', df_val['rating'].value_counts(normalize=True))
print('test', df_test['rating'].value_counts(normalize=True))

print('\nsizes')
print('train', len(df_train))
print('val', len(df_val))
print('test', len(df_test))

proportion counts
train rating
5    0.587422
4    0.200343
3    0.092644
1    0.072018
2    0.047574
Name: proportion, dtype: float64
val rating
5    0.587424
4    0.200342
3    0.092644
1    0.072019
2    0.047571
Name: proportion, dtype: float64
test rating
5    0.587420
4    0.200342
3    0.092644
1    0.072019
2    0.047575
Name: proportion, dtype: float64

sizes
train 1230642
val 263709
test 263709


In [38]:
df_train.to_csv('./data/all-states-train.csv', index=False)
df_val.to_csv('./data/all-states-val.csv', index=False)
df_test.to_csv('./data/all-states-test.csv', index=False)